# Narrative Learning — QLoRA fine-tuning of the story writer

**Goal:** make the small local model (`gemma3:4b`) write complete, syllabus-grounded, age-appropriate lessons as reliably as a 33B teacher model, while still running offline on a school laptop.

**Technique:** knowledge distillation + QLoRA (Quantized Low-Rank Adaptation).

1. `scripts/build_finetune_dataset.py` runs a large teacher model (`gemma4:cloud`, ~33B) through the *real* app pipeline (same retrieval, same prompt) and keeps only answers that pass quality gates (valid JSON, all fields, usable quick-check, 3-paragraph story, definitions that appear in the story).
2. Teachers review `data/finetune/review.csv`; rejected rows are dropped (`--apply-review`).
3. This notebook fine-tunes the student (`gemma-3-4b-it`) on those pairs with QLoRA.
4. The result is exported to GGUF and loaded into Ollama, so the app switches models with one `.env` line.

**Why QLoRA and not full fine-tuning:** full fine-tuning of 4B parameters needs ~60+ GB GPU memory. QLoRA freezes the base model in 4-bit and trains small low-rank adapter matrices (~1% of weights), so it fits a free Colab T4 (16 GB), trains in under an hour, and is much less likely to overfit a few hundred examples or forget general language skills.

Runtime: **Colab → Runtime → Change runtime type → T4 GPU**.

In [ ]:
!pip install -q unsloth
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Upload the dataset
Upload `train.jsonl` and `test.jsonl` from `data/finetune/` (or the `*_approved.jsonl` files after teacher review).

In [ ]:
import json, os
from google.colab import files

uploaded = files.upload()
TRAIN = next(n for n in uploaded if n.startswith('train'))
TEST = next(n for n in uploaded if n.startswith('test'))

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_rows, test_rows = load_jsonl(TRAIN), load_jsonl(TEST)
print(len(train_rows), 'train /', len(test_rows), 'test samples')
print('test topics:', sorted({r['meta']['topic'] for r in test_rows}))

## 2. Load the base model in 4-bit and attach LoRA adapters

| Parameter | Value | Why |
|---|---|---|
| base model | `unsloth/gemma-3-4b-it` | Same model family the app already runs (`gemma3:4b`), so the comparison is fair and deployment is unchanged. |
| `load_in_4bit` | True | The "Q" in QLoRA: frozen base weights in 4-bit NF4 → fits a 16 GB T4. |
| `max_seq_length` | 4096 | Measured: prompt ≈ 1.5k tokens + JSON answer ≤ ~1k tokens ≈ 2.5k; 4096 leaves headroom. |
| `r` (rank) | 16 | Capacity of the adapter. 8 under-fits a structured-output task; 32–64 adds memory and over-fitting risk with a few hundred samples. |
| `lora_alpha` | 16 | Scaling = alpha / r = 1, the standard stable setting. |
| `lora_dropout` | 0 | Unsloth's optimized path; regularisation comes from small r, few epochs and held-out eval instead. |
| target modules | attention + MLP | Attention learns *where* to look in the textbook context; MLP learns the output style/format. |
| vision layers | off | Text-only task. |

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

MAX_SEQ = 4096
model, tokenizer = FastModel.from_pretrained(
    model_name='unsloth/gemma-3-4b-it',
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    random_state=3407,
)
tokenizer = get_chat_template(tokenizer, chat_template='gemma-3')
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

def to_text(rows):
    texts = [tokenizer.apply_chat_template(r['messages'], tokenize=False).removeprefix('<bos>') for r in rows]
    return Dataset.from_dict({'text': texts})

train_ds, eval_ds = to_text(train_rows), to_text(test_rows)
lengths = [len(tokenizer(t)['input_ids']) for t in train_ds['text']]
print('tokens per sample: max', max(lengths), 'mean', sum(lengths) // len(lengths))
print(train_ds[0]['text'][:600])

## 3. Train

| Parameter | Value | Why |
|---|---|---|
| `learning_rate` | 2e-4 | Standard for LoRA (adapters start at zero, need a higher LR than full fine-tuning's ~2e-5). |
| batch 2 × grad-accum 4 | effective batch 8 | Fits T4 memory with ~2.5k-token samples; 8 gives stable gradients. |
| `num_train_epochs` | 2 | Small dataset: 1 epoch under-fits the JSON format, 3+ starts memorising stories (watch eval loss). |
| `warmup_steps` / scheduler | 5 / linear decay | Avoids early large updates; decays to 0 for a clean finish. |
| `optim` | `adamw_8bit` | Same optimiser quality, ~75% less optimiser memory. |
| `weight_decay` | 0.01 | Light regularisation. |
| train on responses only | yes | Loss only on the model's JSON answer, not on the prompt/textbook text it was given. |
| eval on held-out topics | every 10 steps | Test chapters never appear in training → measures generalisation, not memorisation. |

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=5,
        eval_strategy='steps',
        eval_steps=10,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir='outputs',
        report_to='none',
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part='<start_of_turn>user\n',
    response_part='<start_of_turn>model\n',
)
stats = trainer.train()
print(stats)

In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
tr = [(l['step'], l['loss']) for l in log if 'loss' in l]
ev = [(l['step'], l['eval_loss']) for l in log if 'eval_loss' in l]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label='train loss')
if ev:
    plt.plot(*zip(*ev), 'o-', label='eval loss (held-out topics)')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.title('QLoRA training — Narrative story writer')
plt.tight_layout(); plt.savefig('loss_curve.png', dpi=140); plt.show()

## 4. Quick check on held-out topics
Generates for a few test prompts and checks JSON validity. The full comparison against the base model is done locally with `scripts/compare_story_models.py --topic-file data/finetune/test_topics.txt`.

In [ ]:
FastModel.for_inference(model)
REQUIRED = ['science_intro', 'key_definitions', 'key_equations', 'exam_bullets', 'check_question', 'quiz_topic', 'story']
ok = 0
sample = test_rows[:5]
for r in sample:
    prompt = tokenizer.apply_chat_template(r['messages'][:1], tokenize=False, add_generation_prompt=True).removeprefix('<bos>')
    ids = tokenizer([prompt], return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=1800, temperature=0.65, top_p=0.95, top_k=64)
    text = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
    try:
        data = json.loads(text[text.find('{'): text.rfind('}') + 1])
        complete = all(k in data for k in REQUIRED)
    except Exception:
        data, complete = None, False
    ok += complete
    print(r['meta']['topic'], '| complete JSON:', complete, '| words:', len((data or {}).get('story', '').split()))
print(f'{ok}/{len(sample)} complete')

## 5. Export to GGUF for Ollama
Saves the LoRA adapter (small, for the report/repo) and a merged GGUF (for Ollama).

In [ ]:
model.save_pretrained('narrative_lora')
tokenizer.save_pretrained('narrative_lora')
try:
    model.save_pretrained_gguf('narrative_gguf', tokenizer, quantization_method='q4_k_m')
except TypeError:
    model.save_pretrained_gguf('narrative_gguf', quantization_type='Q8_0')
!ls -lh narrative_gguf
!zip -qr narrative_lora.zip narrative_lora loss_curve.png
files.download('narrative_lora.zip')
gguf = [f for f in os.listdir('narrative_gguf') if f.endswith('.gguf')][0]
files.download(os.path.join('narrative_gguf', gguf))

## 6. Use it in the app (on the laptop)

```powershell
# put the downloaded .gguf next to scripts/narrative_lora.Modelfile, edit FROM if the name differs
ollama create narrative-gemma3 -f scripts/narrative_lora.Modelfile

# compare base vs fine-tuned on the held-out chapters
python scripts/compare_story_models.py --models gemma3:4b narrative-gemma3 --topic-file data/finetune/test_topics.txt

# switch the app
# .env: OLLAMA_LLM_MODEL=narrative-gemma3
```

Gemma is provided under and subject to the Gemma Terms of Use found at ai.google.dev/gemma/terms. The fine-tuned model is a Model Derivative under those terms.